# compiles the results from 56 (runner) into a monthly dataset of month accu rain for OBS
- input: aux/rainamountsobs/rain_amounts_{date}.json 
- compile the output to one dataset aux/OBS_monthly_rain_amount
=> output equals output from 55 from ICON!


In [ ]:
import pandas as pd
from pathlib import Path
import glob
import xarray as xr

In [11]:
savedir = Path("../output/aux/rainamountsobs")

files = list(savedir.glob("rain_amounts_*.json"))

df = pd.DataFrame([pd.read_json(f, typ="series") for f in files])

df["rain_amount_mm"] = pd.to_numeric(df["rain_amount_mm"])

# add timeperiod column (false put timeperiod = JUNE eveywhere...)
df = df.drop(columns=["timeperiod"], errors="ignore") # to be save
df.loc[pd.to_datetime(df["date"], format="%Y-%m-%d-%H%M").dt.month == 12, "timeperiod"] = "DECEMBER"
df.loc[pd.to_datetime(df["date"], format="%Y-%m-%d-%H%M").dt.month == 6, "timeperiod"] = "JUNE"

df

,date,rain_amount_mm,timeperiod
0,2023-06-11-1500,0.416528,JUNE
1,2023-12-04-1800,0.431390,DECEMBER
2,2023-12-08-0900,0.388694,DECEMBER
3,2023-12-24-0300,0.419952,DECEMBER
4,2023-06-09-0900,0.411479,JUNE
...,...,...,...
478,2023-12-01-0600,0.413004,DECEMBER
479,2023-12-26-1200,0.360058,DECEMBER
480,2023-06-20-1200,0.389608,JUNE
481,2023-12-30-0000,0.417912,DECEMBER


In [12]:
# group by month and sum
monthly = df.groupby("timeperiod")["rain_amount_mm"].sum()

# get the correct ones for each month
monthly_rain_amount_june = monthly.get("JUNE", 0.0)
monthly_rain_amount_december = monthly.get("DECEMBER", 0.0)

In [14]:
monthly_rain_amount_june,monthly_rain_amount_december

(np.float64(96.63710790127506), np.float64(105.86841157078733))

In [15]:
savedir = Path("../output/aux")
savedir.mkdir(parents=True, exist_ok=True)

monthly_rain_amount = xr.Dataset(
    {
        "rain_amount": ("timeperiod", [monthly_rain_amount_june, monthly_rain_amount_december])
    },
    coords={"timeperiod": ["JUNE", "DECEMBER"]}
)
monthly_rain_amount.attrs = {
    "description": "Monthly domain-wide accumulated rain amount for ICON, area-weighted over the tropical belt domain.",
    "source": "IMERG",
    "units": "mm/month",
    "long_name": "domain-wide accumulated rain amount",
    "method": (
        "Area-weighted spatial mean of 3-hourly mean rain rate (rain_tot_rate), "
        "multiplied by 3 h to obtain mm per "
        "timestep, then summed over all timestamps in the month."
    )
}


monthly_rain_amount.to_netcdf(savedir / "OBS_monthly_rain_amount.nc", engine="h5netcdf")